<a href="https://colab.research.google.com/github/JONATHANPALACIOSParametric/JONATHANPALACIOSParametric/blob/main/Building-Design-Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade openseespy pandas matplotlib plotly -q
print("✓ Instalación de dependencias completada. Ya puedes ejecutar la siguiente celda.")

✓ Instalación de dependencias completada. Ya puedes ejecutar la siguiente celda.


In [2]:
# ================================================================
# OPENSEESPY 3D STRUCTURAL ANALYSIS & DESIGN
# Parametric Steel Building - Robust Execution Version
# ================================================================

import sys
import traceback

try:
    print("1. Iniciando entorno e importación de librerías...")
    import openseespy.opensees as ops
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    from IPython.display import display

    # Asegurar limpieza absoluta de modelos previos
    ops.wipe()
    print("✓ Entorno de OpenSees inicializado y limpio.")

    # ================================================================
    # 1. MODEL INPUTS
    # ================================================================
    print("2. Definiendo parámetros de entrada (Geometría, Materiales, Secciones)...")
    geometry = {
        "n_bays_x": 4,
        "n_bays_y": 3,
        "n_storeys": 4,
        "bay_x": 6.0,       # m
        "bay_y": 5.0,       # m
        "storey_h": 3.5     # m
    }

    materials = pd.DataFrame([
        {
            "mat_id": 1,
            "name": "S355",
            "E": 210000.0,     # MPa
            "G": 80769.0,      # MPa
            "fy": 355.0        # MPa
        }
    ])

    sections = pd.DataFrame([
        {
            "sec_id": 1,
            "name": "HEB 260",
            "type": "COLUMN",
            "A": 11840.0,       # mm2
            "Iy": 51.34e6,    # mm4
            "Iz": 149.2e6,    # mm4
            "J": 1.238e6,     # mm4
            "Wely": 395e3,    # mm3
            "Welz": 1150e3,   # mm3
            "fy": 355.0
        },
        {
            "sec_id": 2,
            "name": "IPE 300",
            "type": "BEAM",
            "A": 5380.0,
            "Iy": 6.04e6,
            "Iz": 83.56e6,
            "J": 0.201e6,
            "Wely": 80.5e3,
            "Welz": 557e3,
            "fy": 355.0
        }
    ])

    # ================================================================
    # 2. GENERATE NODE TABLE
    # ================================================================
    print("3. Generando coordenadas de nodos...")
    nodes = []
    node_map = {}
    node_id = 1

    for k in range(geometry["n_storeys"] + 1):
        z = float(k * geometry["storey_h"])
        for j in range(geometry["n_bays_y"] + 1):
            y = float(j * geometry["bay_y"])
            for i in range(geometry["n_bays_x"] + 1):
                x = float(i * geometry["bay_x"])
                nodes.append({
                    "Node": node_id,
                    "X": x,
                    "Y": y,
                    "Z": z
                })
                node_map[(i, j, k)] = node_id
                node_id += 1

    df_nodes = pd.DataFrame(nodes)
    print(f"✓ Se generaron {len(df_nodes)} nodos.")

    # ================================================================
    # 3. GENERATE MEMBER TABLE
    # ================================================================
    print("4. Creando topología de miembros estructurales...")
    members = []
    member_id = 1

    # Columns
    for k in range(geometry["n_storeys"]):
        for j in range(geometry["n_bays_y"] + 1):
            for i in range(geometry["n_bays_x"] + 1):
                members.append({
                    "Member": member_id,
                    "NodeI": node_map[(i, j, k)],
                    "NodeJ": node_map[(i, j, k + 1)],
                    "Type": "COLUMN",
                    "Section": 1
                })
                member_id += 1

    # Beams X
    for k in range(1, geometry["n_storeys"] + 1):
        for j in range(geometry["n_bays_y"] + 1):
            for i in range(geometry["n_bays_x"]):
                members.append({
                    "Member": member_id,
                    "NodeI": node_map[(i, j, k)],
                    "NodeJ": node_map[(i + 1, j, k)],
                    "Type": "BEAM",
                    "Section": 2
                })
                member_id += 1

    # Beams Y
    for k in range(1, geometry["n_storeys"] + 1):
        for j in range(geometry["n_bays_y"]):
            for i in range(geometry["n_bays_x"] + 1):
                members.append({
                    "Member": member_id,
                    "NodeI": node_map[(i, j, k)],
                    "NodeJ": node_map[(i, j + 1, k)],
                    "Type": "BEAM",
                    "Section": 2
                })
                member_id += 1

    df_members = pd.DataFrame(members)
    print(f"✓ Se generaron {len(df_members)} miembros estructurales.")

    # ================================================================
    # 4. SUPPORTS
    # ================================================================
    df_supports = pd.DataFrame([
        {
            "Node": int(node["Node"]),
            "UX": 1, "UY": 1, "UZ": 1,
            "RX": 1, "RY": 1, "RZ": 1
        }
        for _, node in df_nodes[df_nodes["Z"] == 0.0].iterrows()
    ])

    # ================================================================
    # 5. LOADS
    # ================================================================
    df_loads = pd.DataFrame([
        {
            "Node": int(node["Node"]),
            "Fx": 0.0, "Fy": 0.0, "Fz": -50.0
        }
        for _, node in df_nodes[df_nodes["Z"] > 0.0].iterrows()
    ])

    # ================================================================
    # 6. BUILD OPENSEES MODEL
    # ================================================================
    print("5. Configurando modelo matemático en el motor OpenSees...")
    ops.model("basic", "-ndm", 3, "-ndf", 6)

    # Cargar nodos
    for _, row in df_nodes.iterrows():
        ops.node(int(row["Node"]), float(row["X"]), float(row["Y"]), float(row["Z"]))

    # Aplicar restricciones de soporte
    for _, row in df_supports.iterrows():
        ops.fix(int(row["Node"]), 1, 1, 1, 1, 1, 1)

    # Transformaciones geométricas robustas para vigas y columnas 3D
    ops.geomTransf("Linear", 1, 0.0, 1.0, 0.0) # Columnas (eje longitudinal Z)
    ops.geomTransf("Linear", 2, 0.0, 0.0, 1.0) # Vigas en X
    ops.geomTransf("Linear", 3, 0.0, 0.0, 1.0) # Vigas en Y

    # Agregar elementos estructurales
    for _, member in df_members.iterrows():
        sec = sections[sections["sec_id"] == member["Section"]].iloc[0]

        # Escalar de mm a metros
        A = float(sec["A"]) * 1e-6
        Iy = float(sec["Iy"]) * 1e-12
        Iz = float(sec["Iz"]) * 1e-12
        J = float(sec["J"]) * 1e-12

        E = float(materials.iloc[0]["E"]) * 1e6
        G = float(materials.iloc[0]["G"]) * 1e6

        # Definir la transformación geométrica según corresponda
        if member["Type"] == "COLUMN":
            transf = 1
        else:
            # Determinar si la viga es paralela a X o a Y
            node_i = df_nodes[df_nodes["Node"] == member["NodeI"]].iloc[0]
            node_j = df_nodes[df_nodes["Node"] == member["NodeJ"]].iloc[0]
            if abs(node_i["Y"] - node_j["Y"]) < 1e-3:
                transf = 2
            else:
                transf = 3

        ops.element(
            "elasticBeamColumn",
            int(member["Member"]),
            int(member["NodeI"]),
            int(member["NodeJ"]),
            A, E, G, J, Iy, Iz, transf
        )
    print("✓ Geometría y elementos de OpenSees cargados exitosamente.")

    # ================================================================
    # 7. LOADS AS PATTERN
    # ================================================================
    ops.timeSeries("Linear", 1)
    ops.pattern("Plain", 1, 1)
    for _, load in df_loads.iterrows():
        ops.load(int(load["Node"]), float(load["Fx"]), float(load["Fy"]), float(load["Fz"]), 0.0, 0.0, 0.0)

    # ================================================================
    # 8. RUN STATIC ANALYSIS
    # ================================================================
    print("6. Ejecutando análisis numérico estático...")
    ops.system("BandGeneral")
    ops.numberer("RCM")
    ops.constraints("Plain")
    ops.integrator("LoadControl", 1.0)
    ops.algorithm("Linear")
    ops.analysis("Static")

    ok = ops.analyze(1)
    if ok == 0:
        print("✓ ¡Análisis estático de OpenSees completado con éxito!")
    else:
        raise RuntimeError("La resolución del sistema de ecuaciones falló en OpenSees.")

    # ================================================================
    # 9. PROCESS NODE DISPLACEMENTS
    # ================================================================
    node_results = []
    for _, node in df_nodes.iterrows():
        tag = int(node["Node"])
        disp = ops.nodeDisp(tag)
        node_results.append({
            "Node": tag,
            "X": node["X"], "Y": node["Y"], "Z": node["Z"],
            "UX [mm]": disp[0] * 1000,
            "UY [mm]": disp[1] * 1000,
            "UZ [mm]": disp[2] * 1000,
            "RX": disp[3], "RY": disp[4], "RZ": disp[5]
        })
    df_node_results = pd.DataFrame(node_results)

    # ================================================================
    # 10. PROCESS INTERNAL MEMBER FORCES
    # ================================================================
    member_results = []
    for _, member in df_members.iterrows():
        tag = int(member["Member"])
        force = ops.eleResponse(tag, "localForce")
        if force is None or len(force) < 12:
            continue

        member_results.append({
            "Member": tag,
            "Type": member["Type"],
            "Section": member["Section"],
            "N_i [kN]": force[0] / 1000, "Vy_i [kN]": force[1] / 1000, "Vz_i [kN]": force[2] / 1000,
            "Mx_i [kNm]": force[3] / 1000, "My_i [kNm]": force[4] / 1000, "Mz_i [kNm]": force[5] / 1000,
            "N_j [kN]": force[6] / 1000, "Vy_j [kN]": force[7] / 1000, "Vz_j [kN]": force[8] / 1000,
            "Mx_j [kNm]": force[9] / 1000, "My_j [kNm]": force[10] / 1000, "Mz_j [kNm]": force[11] / 1000
        })
    df_member_results = pd.DataFrame(member_results)

    # ================================================================
    # 11. STEEL DESIGN CHECK
    # ================================================================
    design_results = []
    for _, result in df_member_results.iterrows():
        member = df_members[df_members["Member"] == result["Member"]].iloc[0]
        sec = sections[sections["sec_id"] == member["Section"]].iloc[0]

        NEd = max(abs(result["N_i [kN]"]), abs(result["N_j [kN]"]))
        MyEd = max(abs(result["My_i [kNm]"]), abs(result["My_j [kNm]"]))
        MzEd = max(abs(result["Mz_i [kNm]"]), abs(result["Mz_j [kNm]"]))

        fy = sec["fy"]
        NplRd = float(sec["A"]) * fy / 1000
        MyRd = float(sec["Wely"]) * fy / 1e6
        MzRd = float(sec["Welz"]) * fy / 1e6

        dcr = max(NEd / NplRd, MyEd / MyRd, MzEd / MzRd)
        design_results.append({
            "Member": result["Member"],
            "Section": sec["name"],
            "NEd [kN]": NEd, "MyEd [kNm]": MyEd, "MzEd [kNm]": MzEd,
            "N_Rd [kN]": NplRd, "My_Rd [kNm]": MyRd, "Mz_Rd [kNm]": MzRd,
            "DCR": dcr,
            "Status": "PASS" if dcr <= 1.0 else "FAIL"
        })
    df_design = pd.DataFrame(design_results)

    # ================================================================
    # 12. DISPLAY TABULAR RESULTS
    # ================================================================
    print("\n=== DESPLAZAMIENTOS NODALES MAXYMOS ===")
    display(df_node_results.style.format(precision=2).background_gradient(subset=["UX [mm]", "UY [mm]", "UZ [mm]"]))

    print("\n=== FUERZAS EN LOS MIEMBROS ===")
    display(df_member_results.head(10).style.format(precision=2))

    print("\n=== VERIFICACIÓN DE DISEÑO DE ACERO ===")
    display(df_design.sort_values("DCR", ascending=False).style.format(precision=2).background_gradient(subset=["DCR"]))

    # ================================================================
    # 13. PLOTLY 3D MODEL VISUALIZATION
    # ================================================================
    fig = go.Figure()
    for _, member in df_members.iterrows():
        ni = df_nodes[df_nodes["Node"] == member["NodeI"]].iloc[0]
        nj = df_nodes[df_nodes["Node"] == member["NodeJ"]].iloc[0]
        fig.add_trace(go.Scatter3d(
            x=[ni.X, nj.X], y=[ni.Y, nj.Y], z=[ni.Z, nj.Z],
            mode="lines", line=dict(width=4, color="blue"), showlegend=False
        ))

    fig.update_layout(
        title="Estructura Paramétrica en 3D",
        scene=dict(xaxis_title="X [m]", yaxis_title="Y [m]", zaxis_title="Z [m]", aspectmode="data"),
        height=600
    )
    fig.show()

    # ================================================================
    # 14. STRUCTURAL DESIGN SUMMARY REPORT
    # ================================================================
    print("\n" + "="*60)
    print("STRUCTURAL DESIGN SUMMARY REPORT")
    print("="*60)
    print(f"Nodos del modelo:       {len(df_nodes)}")
    print(f"Miembros cargados:      {len(df_members)}")
    print(f"Número de Niveles:      {geometry['n_storeys']}")
    print(f"Deflexión Máxima Z:     {df_node_results['UZ [mm]'].abs().max():.4f} mm")
    print(f"Ratio de Trabajo Max:   {df_design['DCR'].max():.4f}")
    print(f"Miembros Aptos (PASS):  {(df_design['Status'] == 'PASS').sum()}")
    print(f"Miembros Fallidos:      {(df_design['Status'] == 'FAIL').sum()}")

except Exception as e:
    print("✗ Ocurrió un error inesperado durante el análisis:")
    traceback.print_exc()


1. Iniciando entorno e importación de librerías...
✓ Entorno de OpenSees inicializado y limpio.
2. Definiendo parámetros de entrada (Geometría, Materiales, Secciones)...
3. Generando coordenadas de nodos...
✓ Se generaron 100 nodos.
4. Creando topología de miembros estructurales...
✓ Se generaron 204 miembros estructurales.
5. Configurando modelo matemático en el motor OpenSees...
✓ Geometría y elementos de OpenSees cargados exitosamente.
6. Ejecutando análisis numérico estático...
✓ ¡Análisis estático de OpenSees completado con éxito!

=== DESPLAZAMIENTOS NODALES MAXYMOS ===


,Node,X,Y,Z,UX [mm],UY [mm],UZ [mm],RX,RY,RZ
0,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,2,6.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,3,12.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,4,18.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,5,24.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,6,0.00,5.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
6,7,6.00,5.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
7,8,12.00,5.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
8,9,18.00,5.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,10,24.00,5.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00



=== FUERZAS EN LOS MIEMBROS ===


,Member,Type,Section,N_i [kN],Vy_i [kN],Vz_i [kN],Mx_i [kNm],My_i [kNm],Mz_i [kNm],N_j [kN],Vy_j [kN],Vz_j [kN],Mx_j [kNm],My_j [kNm],Mz_j [kNm]
0,1,COLUMN,1,0.20,0.00,0.00,0.00,0.00,0.00,-0.20,-0.00,-0.00,-0.00,-0.00,-0.00
1,2,COLUMN,1,0.20,0.00,-0.00,0.00,0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,-0.00
2,3,COLUMN,1,0.20,0.00,-0.00,0.00,0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,-0.00
3,4,COLUMN,1,0.20,0.00,-0.00,0.00,-0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,-0.00
4,5,COLUMN,1,0.20,0.00,-0.00,0.00,-0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,0.00
5,6,COLUMN,1,0.20,0.00,0.00,0.00,0.00,0.00,-0.20,-0.00,-0.00,-0.00,-0.00,-0.00
6,7,COLUMN,1,0.20,0.00,-0.00,0.00,0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,-0.00
7,8,COLUMN,1,0.20,0.00,-0.00,0.00,0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,0.00
8,9,COLUMN,1,0.20,0.00,0.00,0.00,-0.00,0.00,-0.20,-0.00,-0.00,-0.00,-0.00,0.00
9,10,COLUMN,1,0.20,0.00,-0.00,0.00,-0.00,0.00,-0.20,-0.00,0.00,-0.00,0.00,0.00



=== VERIFICACIÓN DE DISEÑO DE ACERO ===


,Member,Section,NEd [kN],MyEd [kNm],MzEd [kNm],N_Rd [kN],My_Rd [kNm],Mz_Rd [kNm],DCR,Status
4,5,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
15,16,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
16,17,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
2,3,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
9,10,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
1,2,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
10,11,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
5,6,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
19,20,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS
14,15,HEB 260,0.20,0.00,0.00,4203.20,140.22,408.25,0.00,PASS



STRUCTURAL DESIGN SUMMARY REPORT
Nodos del modelo:       100
Miembros cargados:      204
Número de Niveles:      4
Deflexión Máxima Z:     0.0007 mm
Ratio de Trabajo Max:   0.0000
Miembros Aptos (PASS):  204
Miembros Fallidos:      0
